# Agent Falsifiability Benchmark Suite — Demo Notebook

This notebook demonstrates the **Agent Falsifiability Benchmark Suite** dataset artifact.

## What this artifact does

The `data.py` script constructs a benchmark suite of **10 diverse empirical ML research tasks** spanning both classification and regression domains. Each task includes:

- **Baseline**: A simple model (LogisticRegression/Ridge) performance
- **True Positive**: A methodological improvement (ensemble, scaling, non-linear model) expected to improve performance
- **Negative Control**: A failure condition (permuted labels, noise features, shuffled features) expected to fail
- **Refutation Criteria**: Explicit quantitative thresholds to validate true positives and confirm negative controls
- **Ground Truth**: Known outcome (`success_for_tp_failure_for_nc`)

This notebook loads a curated subset (`mini_demo_data.json`) from the **Iris dataset** (one of the 10 datasets) and demonstrates the data structure.

In [ ]:
# Install dependencies — following aii-colab pattern
# Colab pre-installs: numpy, pandas, scikit-learn, matplotlib, jupyter
# We only install packages NOT pre-installed on Colab

import sys
import subprocess
import importlib

# Packages needed by this artifact (from data.py imports)
required_packages = [
    "numpy",
    "pandas",
    "scikit-learn",
    "matplotlib",
]

def install_if_missing(pkg):
    try:
        importlib.import_module(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg in required_packages:
    install_if_missing(pkg)

# Colab-specific: ensure numpy version compatibility
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Colab has numpy 2.x; some sklearn versions need 1.x compat
    import numpy as np
    if not hasattr(np, 'int'):
        np.int = int
        np.float = float
        np.bool = bool
        np.object = object
        np.str = str
    print("Running in Colab — numpy compat shims applied")
else:
    print("Running locally — using environment as-is")

print("Dependencies ready.")

In [ ]:
# Imports — copy original import block as-is
import json
import numpy as np
import pandas as pd
from sklearn.datasets import (
    fetch_california_housing,
    load_breast_cancer,
    load_diabetes,
    load_wine,
    load_digits,
    load_iris,
    make_classification,
    make_regression
)
import matplotlib.pyplot as plt

print("Imports loaded.")

In [ ]:
# Data loading helper — GitHub URL with local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-1cefba-falsifiable-prediction-graphs-eliminatin/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load mini_demo_data.json from GitHub (after deployment) or local file (now)."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub load failed: {e}")
    
    # Local fallback
    import os
    local_path = "mini_demo_data.json"
    if os.path.exists(local_path):
        with open(local_path) as f:
            return json.load(f)
    
    raise FileNotFoundError(
        "Could not load mini_demo_data.json from GitHub or local path."
    )

In [ ]:
# Load the demo data
data = load_data()
print(f"Loaded {len(data['datasets'])} dataset(s)")
for d in data['datasets']:
    print(f"  - {d['dataset']}: {len(d['examples'])} examples")

## Config — Tunable Parameters

These parameters control the demo scale. Set to minimum values for fast execution.

In [ ]:
# Config cell — ALL tunable parameters defined here

# Data exploration parameters
N_EXAMPLES_TO_SHOW = 5          # Number of examples to display per class
N_FEATURES_TO_SHOW = 4          # Iris has 4 features

# Visualization parameters
PLOT_FIGSIZE = (10, 6)          # Figure size for plots
PLOT_DPI = 100                  # DPI for plots

# Original script parameters (commented out - for reference)
# ORIGINAL_MAX_ROWS_PER_DATASET = 300
# ORIGINAL_SYNTHETIC_SAMPLES = 250
# ORIGINAL_BENCHMARK_SAMPLES = 2000

print("Config loaded.")

## Processing — Explore Dataset Structure

Examine the loaded Iris dataset examples, feature names, and task type.

In [ ]:
# Extract the Iris dataset from loaded data
iris_data = None
for d in data['datasets']:
    if d['dataset'] == 'iris':
        iris_data = d
        break

if iris_data is None:
    raise ValueError("Iris dataset not found in loaded data")

examples = iris_data['examples']
print(f"Dataset: {iris_data['dataset']}")
print(f"Total examples: {len(examples)}")
print(f"Task type: {examples[0]['metadata_task_type']}")
print(f"Feature names: {examples[0]['metadata_feature_names']}")

# Show first few examples
print("\nFirst 3 examples:")
for i, ex in enumerate(examples[:3]):
    print(f"  Example {i}:")
    print(f"    input:  {ex['input']}")
    print(f"    output: {ex['output']}")
    print(f"    row_index: {ex['metadata_row_index']}")

## Processing — Parse Features and Labels

Convert JSON string inputs to numeric feature matrices and extract class labels.

In [ ]:
# Parse features and labels from examples
feature_names = examples[0]['metadata_feature_names']
n_features = len(feature_names)

X_list = []
y_list = []
row_indices = []

for ex in examples:
    # Parse input JSON string to dict
    row_dict = json.loads(ex['input'])
    # Ensure consistent feature ordering
    row_values = [row_dict[fn] for fn in feature_names]
    X_list.append(row_values)
    y_list.append(int(ex['output']))
    row_indices.append(ex['metadata_row_index'])

X = np.array(X_list)
y = np.array(y_list)
row_indices = np.array(row_indices)

print(f"Feature matrix shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Unique classes: {np.unique(y)}")
print(f"Class distribution: {np.bincount(y)}")
print(f"Row indices range: {row_indices.min()} - {row_indices.max()}")

## Processing — Basic Statistics

Compute per-class feature statistics (mean, std) to understand the data distribution.

In [ ]:
# Compute per-class statistics
classes = np.unique(y)
class_names = ['Setosa', 'Versicolor', 'Virginica']

print("Per-class feature means:")
for c in classes:
    mask = (y == c)
    mean_vals = X[mask].mean(axis=0)
    std_vals = X[mask].std(axis=0)
    print(f"  Class {c} ({class_names[c]}): n={mask.sum()}")
    for fn, mv, sv in zip(feature_names, mean_vals, std_vals):
        print(f"    {fn}: mean={mv:.3f}, std={sv:.3f}")

## Results — Visualization

Create pair plots and feature distribution plots to visualize the Iris dataset structure.

In [ ]:
# Visualization: Feature distributions by class
fig, axes = plt.subplots(2, 2, figsize=PLOT_FIGSIZE, dpi=PLOT_DPI)
axes = axes.flatten()
colors = ['tab:blue', 'tab:orange', 'tab:green']

for i, fn in enumerate(feature_names):
    ax = axes[i]
    for c in classes:
        mask = (y == c)
        ax.hist(X[mask, i], bins=15, alpha=0.6, label=class_names[c], color=colors[c], density=True)
    ax.set_xlabel(fn)
    ax.set_ylabel('Density')
    ax.set_title(f'{fn} by Class')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Visualization: Pairwise scatter plot (first 2 features for simplicity)
fig, ax = plt.subplots(figsize=(8, 6), dpi=PLOT_DPI)

for c in classes:
    mask = (y == c)
    ax.scatter(X[mask, 0], X[mask, 1], 
               label=class_names[c], color=colors[c], alpha=0.7, s=50)

ax.set_xlabel(feature_names[0])
ax.set_ylabel(feature_names[1])
ax.set_title('Iris: Sepal Length vs Sepal Width')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

This demo notebook demonstrates the **Agent Falsifiability Benchmark Suite** dataset structure using a curated subset of the Iris dataset (30 examples, 10 per class).

### Key Points:
- The full artifact generates **10 datasets** (6 real + 4 synthetic) with **baseline, true positive, and negative control** conditions
- Each dataset includes explicit **refutation criteria** for automated falsification evaluation
- This mini demo shows **one dataset (Iris)** with **3 classes, 4 features, 30 samples**
- The data loading pattern uses **GitHub URL with local fallback** for Colab compatibility

### Next Steps:
- Run the full `data.py` to generate all 10 datasets with 300 rows each
- Run `build_dataset.py` to generate the full benchmark with ML model evaluations
- Use the benchmark to evaluate agent planning and negative result detection